# Production RAG Systems

**Module 10: Retrieval-Augmented Generation**  
**Objective**: Build production-ready RAG applications

Production RAG with modern tools:
- LangChain framework
- Vector stores (Chroma, Pinecone, Weaviate)
- Document loaders and processing  
- Query optimization
- Evaluation frameworks

## What You'll Learn
1. LangChain RAG pipeline
2. Integrating vector databases
3. Document processing at scale
4. Advanced retrieval strategies
5. Query rewriting and routing
6. Evaluating RAG systems

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

torch.manual_seed(42)

## 1. LangChain RAG Pipeline

In [ ]:
# LangChain RAG examples

langchain_examples = """
# ================================================
# Installation
# ================================================
# pip install langchain langchain-community chromadb sentence-transformers

# ================================================
# Basic RAG Pipeline
# ================================================
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.llms import HuggingFacePipeline

# 1. Load documents
loader = TextLoader("documents.txt")
documents = loader.load()

# 2. Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\\n\\n", "\\n", ". ", " ", ""]
)
chunks = text_splitter.split_documents(documents)

# 3. Create embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# 4. Build vector store  
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

# 5. Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}  # Top 3 documents
)

# 6. Create QA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",  # Stuff all docs into prompt
    retriever=retriever,
    return_source_documents=True
)

# 7. Query
result = qa_chain({"query": "What is machine learning?"})
print(result["result"])
print(f"Source: {result['source_documents'][0].metadata}")


# ================================================
# Advanced Retrieval: MMR (Maximum Marginal Relevance)
# ================================================
# Balances relevance with diversity

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 20,  # Fetch 20, return diverse 5
        "lambda_mult": 0.7  # 0=max diversity, 1=max relevance
    }
)


# ================================================
# Contextual Compression (Filter irrelevant parts)
# ================================================
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever
)

# Only relevant parts of documents are returned


# ================================================
# Multi-Query Retriever (Generate multiple query variations)
# ================================================
from langchain.retrievers.multi_query import MultiQueryRetriever

retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(),
    llm=llm
)

# Automatically generates variations:
# "What is ML?" → ["What is machine learning?", "Define ML", "Explain ML"]


# ================================================
# Parent Document Retriever (Retrieve small, return large)
# ================================================
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore

# Embed small chunks, but return full parent documents
store = InMemoryStore()
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=small_splitter,  # For indexing
    parent_splitter=large_splitter  # For retrieval
)


# ================================================
# Self-Query Retriever (Metadata filtering)
# ================================================
from langchain.retrievers.self_query.base import SelfQueryRetriever

metadata_field_info = [
    {"name": "source", "description": "The source file", "type": "string"},
    {"name": "page", "description": "Page number", "type": "integer"},
]

retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents="Technical documentation",
    metadata_field_info=metadata_field_info
)

# Query: "What did the CEO say about revenue in Q4 2023 earnings?"
# Automatically extracts metadata filter: source="Q4_2023_earnings.pdf"
"""

print("LangChain RAG Patterns:\n")
print(langchain_examples)

## 2. Vector Store Integrations

In [ ]:
vector_store_comparison = """
# ================================================
# ChromaDB (Local, open-source)
# ================================================
from langchain.vectorstores import Chroma

vectorstore = Chroma(
    embedding_function=embeddings,
    persist_directory="./chroma_db"
)

# Pros: Free, local, simple setup
# Cons: Not for production scale (millions of vectors)


# ================================================
# Pinecone (Managed, cloud)
# ================================================
import pinecone
from langchain.vectorstores import Pinecone

pinecone.init(api_key="your-key", environment="us-west1-gcp")
index = pinecone.Index("my-index")

vectorstore = Pinecone(index, embeddings, "text")

# Pros: Managed, scales to billions, fast
# Cons: Paid service


# ================================================
# Weaviate (Hybrid search, self-hosted or cloud)
# ================================================
import weaviate
from langchain.vectorstores import Weaviate

client = weaviate.Client("http://localhost:8080")

vectorstore = Weaviate(
    client=client,
    embedding=embeddings,
    by_text=False
)

# Pros: Hybrid search (vector + keyword), GraphQL API
# Cons: More complex setup


# ================================================
# Qdrant (High performance, open-source)
# ================================================
from qdrant_client import QdrantClient
from langchain.vectorstores import Qdrant

client = QdrantClient(host="localhost", port=6333)

vectorstore = Qdrant(
    client=client,
    collection_name="my_collection",
    embeddings=embeddings
)

# Pros: Fast, open-source, good for self-hosting
# Cons: Less ecosystem than Pinecone


# ================================================
# FAISS (Facebook AI Similarity Search - Local)
# ================================================
from langchain.vectorstores import FAISS

vectorstore = FAISS.from_documents(documents, embeddings)

# Save/load
vectorstore.save_local("faiss_index")
vectorstore = FAISS.load_local("faiss_index", embeddings)

# Pros: Extremely fast, many indexing algorithms
# Cons: In-memory only, no persistence features


# ================================================
# Comparison Matrix
# ================================================
| Vector Store | Scale      | Hosting    | Cost   | Speed | Hybrid Search |
|--------------|------------|------------|--------|-------|---------------|
| ChromaDB     | Small      | Local      | Free   | Fast  | No            |
| Pinecone     | Billion+   | Cloud      | Paid   | Fast  | Coming soon   |
| Weaviate     | Million+   | Both       | Mixed  | Fast  | Yes           |
| Qdrant       | Million+   | Both       | Free   | Fast  | Yes           |
| FAISS        | Million+   | Local      | Free   | Ultra | No            |

Recommendations:
- Development: ChromaDB or FAISS
- Production (managed): Pinecone
- Production (self-hosted): Qdrant or Weaviate
- Maximum performance: FAISS (if fits in RAM)
"""

print("Vector Store Options:\n")
print(vector_store_comparison)

## 3. Document Loaders and Processing

In [ ]:
document_processing = """
# ================================================
# Document Loaders (100+ types supported)
# ================================================

# PDF
from langchain.document_loaders import PyPDFLoader
loader = PyPDFLoader("document.pdf")

# Word documents
from langchain.document_loaders import Docx2txtLoader
loader = Docx2txtLoader("document.docx")

# Web pages
from langchain.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://example.com")

# CSV
from langchain.document_loaders import CSVLoader
loader = CSVLoader("data.csv")

# JSON
from langchain.document_loaders import JSONLoader
loader = JSONLoader("data.json", jq_schema=".content")

# Directory (recursive)
from langchain.document_loaders import DirectoryLoader
loader = DirectoryLoader("./docs", glob="**/*.pdf")

# Google Drive
from langchain.document_loaders import GoogleDriveLoader
loader = GoogleDriveLoader(folder_id="folder-id")

# Notion
from langchain.document_loaders import NotionDirectoryLoader
loader = NotionDirectoryLoader("notion_export")

# YouTube transcripts
from langchain.document_loaders import YoutubeLoader
loader = YoutubeLoader.from_youtube_url("https://youtube.com/watch?v=...")


# ================================================
# Text Splitters
# ================================================

# Recursive Character (Recommended)
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\\n\\n", "\\n", ". ", " ", ""]
)

# Token-based (respects token limits)
from langchain.text_splitter import TokenTextSplitter

splitter = TokenTextSplitter(
    chunk_size=512,
    chunk_overlap=50
)

# Markdown-aware
from langchain.text_splitter import MarkdownTextSplitter

splitter = MarkdownTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

# Code-aware (respects function boundaries)
from langchain.text_splitter import (
    RecursiveCharacterTextSplitter,
    Language
)

python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=500,
    chunk_overlap=50
)


# ================================================
# Document Transformers (Clean/Enrich)
# ================================================

# Remove duplicates
from langchain.document_transformers import EmbeddingsRedundantFilter

redundant_filter = EmbeddingsRedundantFilter(embeddings=embeddings)
filtered_docs = redundant_filter.transform_documents(documents)

# Extract metadata
from langchain.document_transformers import Html2TextTransformer

transformer = Html2TextTransformer()
text_docs = transformer.transform_documents(html_docs)

# Generate summaries for each chunk
from langchain.chains.summarize import load_summarize_chain

chain = load_summarize_chain(llm, chain_type="map_reduce")
summaries = chain.run(documents)
"""

print("Document Processing:\n")
print(document_processing)

## 4. Advanced Retrieval Patterns

In [ ]:
def visualize_retrieval_patterns():
    """Visualize different retrieval strategies."""
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. Standard Retrieval
    ax = axes[0, 0]
    query_pos = (0.5, 0.9)
    doc_positions = [(0.3, 0.5), (0.5, 0.4), (0.7, 0.6), (0.2, 0.3), (0.8, 0.3)]
    
    # Draw query
    circle = plt.Circle(query_pos, 0.08, color='red', ec='black', linewidth=2)
    ax.add_patch(circle)
    ax.text(*query_pos, 'Q', ha='center', va='center', fontsize=12, weight='bold')
    
    # Draw documents with similarity scores
    scores = [0.9, 0.85, 0.75, 0.6, 0.5]
    for i, (pos, score) in enumerate(zip(doc_positions, scores)):
        color = 'lightgreen' if i < 3 else 'lightgray'
        circle = plt.Circle(pos, 0.06, color=color, ec='black', linewidth=1)
        ax.add_patch(circle)
        ax.text(*pos, f'D{i+1}', ha='center', va='center', fontsize=9)
        
        # Draw line to query
        if i < 3:
            ax.plot([query_pos[0], pos[0]], [query_pos[1], pos[1]], 
                   'k--', alpha=0.3)
            # Add score
            mid_x = (query_pos[0] + pos[0]) / 2
            mid_y = (query_pos[1] + pos[1]) / 2
            ax.text(mid_x, mid_y, f'{score:.2f}', fontsize=8, 
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title('Standard Retrieval\n(Top-3 by similarity)', fontsize=11, weight='bold')
    ax.axis('off')
    
    # 2. MMR (Maximum Marginal Relevance)
    ax = axes[0, 1]
    doc_positions = [(0.32, 0.5), (0.35, 0.48), (0.6, 0.3), (0.25, 0.7), (0.75, 0.6)]
    scores = [0.9, 0.88, 0.75, 0.7, 0.72]
    
    circle = plt.Circle(query_pos, 0.08, color='red', ec='black', linewidth=2)
    ax.add_patch(circle)
    ax.text(*query_pos, 'Q', ha='center', va='center', fontsize=12, weight='bold')
    
    # MMR selects diverse docs (not just top scores)
    selected = [0, 2, 4]  # D1 (most similar), D3 (diverse), D5 (diverse)
    for i, (pos, score) in enumerate(zip(doc_positions, scores)):
        color = 'lightblue' if i in selected else 'lightgray'
        circle = plt.Circle(pos, 0.06, color=color, ec='black', linewidth=1)
        ax.add_patch(circle)
        ax.text(*pos, f'D{i+1}', ha='center', va='center', fontsize=9)
        
        if i in selected:
            ax.plot([query_pos[0], pos[0]], [query_pos[1], pos[1]], 
                   'b--', alpha=0.3)
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title('MMR Retrieval\n(Balances relevance + diversity)', fontsize=11, weight='bold')
    ax.axis('off')
    ax.text(0.5, 0.05, 'Avoids redundant docs!', ha='center', fontsize=9, 
           style='italic', bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))
    
    # 3. Multi-Query
    ax = axes[1, 0]
    queries = [(0.4, 0.9), (0.5, 0.87), (0.6, 0.9)]
    doc_positions = [(0.3, 0.5), (0.5, 0.4), (0.7, 0.6), (0.2, 0.3), (0.8, 0.3)]
    
    # Multiple query variations
    for i, qpos in enumerate(queries):
        circle = plt.Circle(qpos, 0.06, color='red', ec='black', linewidth=1.5, alpha=0.7)
        ax.add_patch(circle)
        ax.text(*qpos, f'Q{i+1}', ha='center', va='center', fontsize=9)
    
    # Docs retrieved by any query
    for i, pos in enumerate(doc_positions):
        color = 'lightcoral' if i < 4 else 'lightgray'
        circle = plt.Circle(pos, 0.06, color=color, ec='black', linewidth=1)
        ax.add_patch(circle)
        ax.text(*pos, f'D{i+1}', ha='center', va='center', fontsize=9)
        
        if i < 4:
            # Connect to relevant query
            q_idx = i % 3
            ax.plot([queries[q_idx][0], pos[0]], [queries[q_idx][1], pos[1]], 
                   'r--', alpha=0.2)
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title('Multi-Query Retrieval\n(Generate query variations)', fontsize=11, weight='bold')
    ax.axis('off')
    ax.text(0.5, 0.8, 'Q1: "What is ML?"\\nQ2: "Define ML"\\nQ3: "Explain ML"', 
           ha='center', fontsize=8, style='italic',
           bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))
    
    # 4. Contextual Compression
    ax = axes[1, 1]
    doc_pos = (0.3, 0.5)
    
    # Large document
    rect = plt.Rectangle((0.2, 0.3), 0.2, 0.4, color='lightgray', ec='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(0.3, 0.25, 'Full Document\\n(1000 tokens)', ha='center', fontsize=8)
    
    # Compressed/relevant part
    rect = plt.Rectangle((0.24, 0.42), 0.12, 0.15, color='lightgreen', ec='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(0.3, 0.495, 'Relevant\\nPart', ha='center', fontsize=7, weight='bold')
    
    # Arrow to LLM
    ax.annotate('', xy=(0.7, 0.5), xytext=(0.42, 0.5),
               arrowprops=dict(arrowstyle='->', lw=3, color='green'))
    
    # LLM
    circle = plt.Circle((0.75, 0.5), 0.1, color='plum', ec='black', linewidth=2)
    ax.add_patch(circle)
    ax.text(0.75, 0.5, 'LLM', ha='center', va='center', fontsize=11, weight='bold')
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title('Contextual Compression\n(Extract only relevant parts)', fontsize=11, weight='bold')
    ax.axis('off')
    ax.text(0.5, 0.1, 'Saves tokens, improves focus!', ha='center', fontsize=9,
           style='italic', bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))
    
    plt.tight_layout()
    plt.show()
    
    print("\nRetrieval Pattern Comparison:")
    print("="*70)
    print("\n1. STANDARD RETRIEVAL")
    print("   • Returns top-k by similarity score")
    print("   • Problem: May retrieve redundant/similar documents")
    
    print("\n2. MMR (Maximum Marginal Relevance)")
    print("   • Balances relevance with diversity")
    print("   • λ parameter: 0=max diversity, 1=max relevance")
    print("   • Use: Avoid redundant information")
    
    print("\n3. MULTI-QUERY")
    print("   • LLM generates multiple query variations")
    print("   • Retrieves using all variations")
    print("   • Use: Improve recall, handle ambiguous queries")
    
    print("\n4. CONTEXTUAL COMPRESSION")
    print("   • LLM extracts only relevant parts of retrieved docs")
    print("   • Reduces token usage, improves precision")
    print("   • Use: Long documents, need to save tokens")
    print("="*70)

visualize_retrieval_patterns()

## 5. Evaluation Frameworks

In [ ]:
evaluation_guide = """
# ================================================
# RAG Evaluation Metrics
# ================================================

# 1. RETRIEVAL METRICS (How good is document retrieval?)
# ----------------------------------------------------

# Precision@K: % of retrieved docs that are relevant
precision_at_k = relevant_retrieved / k

# Recall@K: % of all relevant docs that were retrieved
recall_at_k = relevant_retrieved / total_relevant

# MRR (Mean Reciprocal Rank): Position of first relevant doc
mrr = 1 / rank_of_first_relevant

# NDCG (Normalized Discounted Cumulative Gain): Ranking quality
# Accounts for position - higher rank = more weight
import numpy as np

def dcg_at_k(relevances, k):
    relevances = np.asarray(relevances)[:k]
    return np.sum(relevances / np.log2(np.arange(2, relevances.size + 2)))

def ndcg_at_k(relevances, k):
    dcg = dcg_at_k(relevances, k)
    idcg = dcg_at_k(sorted(relevances, reverse=True), k)
    return dcg / idcg if idcg > 0 else 0


# 2. GENERATION METRICS (How good are the answers?)
# ------------------------------------------------------

# Faithfulness: Is answer grounded in retrieved docs?
# Use LLM to check if answer follows from context

from langchain.evaluation import load_evaluator

evaluator = load_evaluator("context_qa")
result = evaluator.evaluate_strings(
    prediction="Paris is the capital of France",
    input="What is the capital of France?",
    reference="Paris is the capital and largest city of France..."
)
# Returns: score (0-1), reasoning

# Answer Relevance: Does answer address the question?
evaluator = load_evaluator("qa")
result = evaluator.evaluate_strings(
    prediction="Paris is the capital of France",
    input="What is the capital of France?"
)

# Context Relevance: Are retrieved docs relevant to query?
evaluator = load_evaluator("context_qa")  


# 3. END-TO-END EVALUATION FRAMEWORKS
# ----------------------------------------

# RAGAS (RAG Assessment)
# Comprehensive RAG evaluation
from ragas import evaluate
from ragas.metrics import (
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy
)

result = evaluate(
    dataset,
    metrics=[
        context_precision,
        context_recall,
        faithfulness,
        answer_relevancy
    ]
)

# TruLens (Observable RAG)
# Track and evaluate RAG apps
from trulens_eval import TruChain, Feedback

# Define feedback functions
f_groundedness = Feedback(llm).groundedness()
f_answer_relevance = Feedback(llm).relevance()
f_context_relevance = Feedback(llm).qs_relevance()

# Wrap chain
tru_recorder = TruChain(
    qa_chain,
    app_id="my_rag_app",
    feedbacks=[f_groundedness, f_answer_relevance, f_context_relevance]
)

# Run and track
with tru_recorder as recording:
    qa_chain.run("What is machine learning?")

# View in dashboard
tru.run_dashboard()


# 4. A/B TESTING PATTERNS
# -------------------------

import random

def ab_test_retrievers(query, retriever_a, retriever_b, n_trials=100):
    results = {"a_wins": 0, "b_wins": 0, "ties": 0}
    
    for _ in range(n_trials):
        # Get results from both
        docs_a = retriever_a.get_relevant_documents(query)
        docs_b = retriever_b.get_relevant_documents(query)
        
        # Use LLM as judge
        judge_prompt = f\"\"\"
        Which set of documents is more relevant to: "{query}"?
        
        Set A: {docs_a}
        Set B: {docs_b}
        
        Answer: A, B, or Tie
        \"\"\"
        
        verdict = llm(judge_prompt).strip()
        
        if "A" in verdict:
            results["a_wins"] += 1
        elif "B" in verdict:
            results["b_wins"] += 1
        else:
            results["ties"] += 1
    
    return results


# 5. MONITORING IN PRODUCTION
# -------------------------------

# Log all queries and responses
import datetime

def log_rag_interaction(query, retrieved_docs, answer, feedback=None):
    log_entry = {
        "timestamp": datetime.datetime.now().isoformat(),
        "query": query,
        "retrieved_doc_ids": [doc.metadata.get("id") for doc in retrieved_docs],
        "answer": answer,
        "user_feedback": feedback,  # Thumbs up/down
        "retrieval_scores": [doc.metadata.get("score") for doc in retrieved_docs]
    }
    
    # Save to database/file
    with open("rag_logs.jsonl", "a") as f:
        f.write(json.dumps(log_entry) + "\\n")

# Analyze later for:
# - Low retrieval scores (poor matches)
# - Negative feedback (bad answers)
# - Common failure patterns
"""

print("RAG Evaluation Guide:\n")
print(evaluation_guide)

# Visualize metrics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Retrieval metrics comparison
metrics = ['Precision@3', 'Recall@3', 'NDCG@3', 'MRR']
retriever_a = [0.85, 0.60, 0.78, 0.82]
retriever_b = [0.75, 0.75, 0.82, 0.70]

x = np.arange(len(metrics))
width = 0.35

axes[0].bar(x - width/2, retriever_a, width, label='Retriever A', color='skyblue')
axes[0].bar(x + width/2, retriever_b, width, label='Retriever B', color='lightcoral')
axes[0].set_ylabel('Score')
axes[0].set_title('Retrieval Metrics Comparison')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].legend()
axes[0].set_ylim(0, 1)
axes[0].grid(axis='y', alpha=0.3)

# Generation metrics
gen_metrics = ['Faithfulness', 'Relevance', 'Coherence']
scores = [0.88, 0.92, 0.85]

axes[1].barh(gen_metrics, scores, color='lightgreen')
axes[1].set_xlabel('Score')
axes[1].set_title('Generation Quality Metrics ')
axes[1].set_xlim(0, 1)
axes[1].grid(axis='x', alpha=0.3)

# Add score labels
for i, (metric, score) in enumerate(zip(gen_metrics, scores)):
    axes[1].text(score + 0.02, i, f'{score:.2f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

## 6. Production Best Practices

In [ ]:
production_checklist = """
# ================================================
# PRODUCTION RAG CHECKLIST
# ================================================

✅ 1. DOCUMENT PROCESSING
   • Implement incremental indexing (only new/changed docs)
   • Handle different file formats gracefully
   • Extract and preserve metadata (source, date, author)
   • Implement deduplication
   • Version control for document corpus

✅ 2. EMBEDDING & INDEXING
   • Batch process documents (don't embed one-by-one)
   • Use consistent embedding model (don't change mid-stream)
   • Normalize all embeddings
   • Implement periodic re-indexing
   • Monitor index size and performance

✅ 3. RETRIEVAL OPTIMIZATION
   • Tune chunk size for your domain (test 256/512/1024)
   • Use hybrid search (dense + sparse)
   • Implement re-ranking (cross-encoder)
   • Add metadata filtering
   • Cache frequent queries

✅ 4. GENERATION
   • Set appropriate temperature (0.1-0.3 for factual)
   • Implement prompt templates
   • Add citations/sources in response
   • Handle cases with no relevant docs found
   • Limit response length

✅ 5. MONITORING & EVALUATION
   • Log all queries and responses
   • Track retrieval scores
   • Collect user feedback (thumbs up/down)
   • Monitor latency (p50, p95, p99)
   • A/B test changes

✅ 6. ERROR HANDLING
   • Fallback if vector DB unavailable
   • Handle empty retrieval results
   • Timeout for slow queries
   • Rate limiting
   • Graceful degradation

✅ 7. SECURITY & PRIVACY
   • Implement access control on documents
   • Don't expose full document paths
   • Sanitize queries (prevent injection)
   • PII detection and masking
   • Audit logs

✅ 8. SCALABILITY
   • Use managed vector DB for production
   • Implement request queuing
   • Horizontal scaling for embeddings
   • Caching layer (Redis)
   • CDN for static content

✅ 9. COST OPTIMIZATION
   • Cache embeddings (don't re-compute)
   • Use cheaper models where possible
   • Batch requests
   • Monitor API usage
   • Implement tiered retrieval (fast→slow)

✅ 10. CONTINUOUS IMPROVEMENT
   • Analyze failed queries
   • Update document corpus regularly
   • Retrain/update embedding model
   • Fine-tune generation model
   • Iterate on chunking strategy


# ================================================
# EXAMPLE: Production RAG Class
# ================================================

from functools import lru_cache
import time
from contextlib import contextmanager

class ProductionRAG:
    def __init__(self, vectorstore, llm, cache_size=1000):
        self.vectorstore = vectorstore
        self.llm = llm
        self.query_cache = {}
        self.cache_size = cache_size
    
    @contextmanager
    def timer(self, operation):
        start = time.time()
        yield
        duration = time.time() - start
        self.log_metric(f"{operation}_duration", duration)
    
    def log_metric(self, metric_name, value):
        # Send to monitoring (Prometheus, CloudWatch, etc.)
        print(f"METRIC: {metric_name}={value:.4f}")
    
    def retrieve(self, query, k=3):
        # Check cache
        cache_key = f"{query}_{k}"
        if cache_key in self.query_cache:
            self.log_metric("cache_hit", 1)
            return self.query_cache[cache_key]
        
        # Retrieve with timing
        with self.timer("retrieval"):
            docs = self.vectorstore.similarity_search(query, k=k)
        
        # Cache results
        if len(self.query_cache) >= self.cache_size:
            # Evict oldest
            self.query_cache.pop(next(iter(self.query_cache)))
        self.query_cache[cache_key] = docs
        
        # Log retrieval quality
        if docs:
            avg_score = sum(doc.metadata.get('score', 0) for doc in docs) / len(docs)
            self.log_metric("retrieval_avg_score", avg_score)
        
        return docs
    
    def generate(self, query, docs):
        if not docs:
            return "I couldn't find relevant information to answer your question."
        
        # Build context
        context = "\\n\\n".join([doc.page_content for doc in docs])
        
        # Generate with timing
        with self.timer("generation"):
            prompt = f\"\"\"Answer the question based on the context below.
            If you cannot answer based on the context, say so.
            
            Context: {context}
            
            Question: {query}
            
            Answer:\"\"\"
            
            answer = self.llm(prompt)
        
        # Add citations
        sources = [doc.metadata.get('source', 'Unknown') for doc in docs]
        answer += f"\\n\\nSources: {', '.join(set(sources))}"
        
        return answer
    
    def query(self, user_query, user_id=None):
        try:
            # Retrieve
            docs = self.retrieve(user_query)
            
            # Generate
            answer = self.generate(user_query, docs)
            
            # Log interaction
            self.log_interaction(user_query, docs, answer, user_id)
            
            return answer
            
        except Exception as e:
            self.log_metric("error", 1)
            self.log_error(str(e))
            return "I'm sorry, I encountered an error. Please try again."
    
    def log_interaction(self, query, docs, answer, user_id):
        # Log to database for analysis
        pass
    
    def log_error(self, error_msg):
        # Send to error tracking (Sentry, etc.)
        print(f"ERROR: {error_msg}")
"""

print("Production Best Practices:\n")
print(production_checklist)

## Summary

You've mastered production RAG systems:
- ✅ LangChain RAG pipeline
- ✅ Vector store integrations (Chroma, Pinecone, FAISS, Weaviate, Qdrant)
- ✅ Document loaders (100+ types)
- ✅ Advanced retrieval (MMR, Multi-Query, Compression)
- ✅ Evaluation frameworks (RAGAS, TruLens)
- ✅ Production best practices

**Key Insights**:
1. **LangChain** provides batteries-included RAG framework
2. **Vector stores**: ChromaDB (dev), Pinecone (prod-managed), Qdrant (prod-self-hosted)
3. **Retrieval patterns**: Standard → MMR → Multi-Query → Contextual Compression
4. **Evaluation**: Track retrieval (precision, recall, NDCG) + generation (faithfulness, relevance)
5. **Production**: Caching, monitoring, error handling, continuous improvement

**Best Practices**:
1. Start with RecursiveCharacterTextSplitter (chunk_size=500, overlap=50)
2. Use hybrid search (vector + BM25)
3. Implement re-ranking with cross-encoder
4. Log everything: queries, retrievals, responses, feedback
5. A/B test retrieval strategies
6. Monitor latency and cost
7. Update documents incrementally, not full re-index
8. Add citations/sources to responses

**Production Deployment**:
```
Development: LangChain + ChromaDB + HuggingFace embeddings
Production: LangChain + Pinecone/Qdrant + OpenAI embeddings
```

**Next Module**: AI Agents and Model Context Protocol (MCP)

## Exercises

1. **Build multi-modal RAG**: Add image search with CLIP embeddings
2. **Implement query routing**: Route questions to appropriate databases
3. **Create conversational RAG**: Add chat history and follow-up questions